# Setup

## Package Installs

In [0]:
%pip install catboost lightgbm optuna duckdb tabulate shap

## Imports

In [0]:
import numpy as np
import pandas as pd
import duckdb
from tabulate import tabulate

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_log_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

## Global Vairables

In [0]:
def check_environment() -> str:
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

In [0]:
TARGET: str = "is_fraud"
N_SPLITS: int = 5
SEED: int = 42
DRIVE_FOLDER_NAME: str = "Credit Card Fraud"
ENVIRONMENT: str = check_environment()

non_relevant_columns: list[str] = ['id', 'source']
eps: float = 1e-5

## Helper Functions

### Load Data

In [0]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/train.csv')
    test_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

### Split Data

In [0]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

In [0]:
train_df, test_df, df = load_data(ENVIRONMENT)

# EDA

## Overview

### Len (999,944)

In [0]:
len(df)

### Info

In [0]:
df.info()

### Null counts

In [0]:
df.isna().sum()

### Describe

In [0]:
df.describe().T

In [0]:
df.describe(include='object').T

## Categorical features

### Fraud rate + counts

In [0]:
import matplotlib.ticker as mtick

cat_cols = list(df.select_dtypes(include=["object", "bool"]).columns)+ ['day_of_week']
cat_cols.remove('source')
n_cols = 3
n_rows = int(np.ceil(len(cat_cols) / n_cols))

fig, axs = plt.subplots(ncols=n_cols, nrows=n_rows, figsize=(6 * n_cols, 4 * n_rows))
for ax, col in zip(axs.flat, cat_cols):
    mean_order = (
        df.groupby(by=f"{col}")[TARGET]
        .mean()
        .sort_values(ascending=False)
        .index
    )

    sns.countplot(
        data=df, x=f"{col}", ax=ax, color="lightgrey", order=mean_order
    )

    ax_1 = ax.twinx()
    sns.pointplot(
        data=df,
        x=f"{col}",
        y=TARGET,
        color="crimson",
        ax=ax_1,
        err_kws={"linewidth": 1},
        linewidth=2,
        markersize=5,
        linestyles='none'
    )

    ax_1.axhline(
        y=df[TARGET].mean(),
        ls="--",
        color="crimson",
        alpha=0.8,
        label="overall fraud rate",
    )
    ax_1.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(f"{col}")
    ax.tick_params(axis="x", rotation=45)


plt.tight_layout()

## Continuous features

### Fraud rate + histogram

In [0]:
import pandas as pd
import matplotlib.ticker as mtick

target = 'is_fraud'
num_cols = ['amount_usd', 'hours_since_last_txn', 'distance_from_home_km',
            'card_age_months', 'customer_age', 'account_balance_usd',
            'velocity_score', 'merchant_risk_score']
base_rate = df[target].mean()

fig, axs = plt.subplots(len(num_cols), 2, figsize=(12, 3.5 * len(num_cols)))

for (ax_d, ax_r), col in zip(axs, num_cols):
    # 1) distribution per class, each normalised to its own total
    sns.histplot(data=df, x=col, hue=target, stat='density', common_norm=False,
                 element='step', fill=False, bins=50, ax=ax_d)
    ax_d.set_title(f'{col} – distribution by class')

    # 2) fraud rate across 20 equal-count bins
    bins = pd.qcut(df[col], q=20, duplicates='drop')
    g = df.groupby(bins, observed=True).agg(x=(col, 'median'), rate=(target, 'mean'))
    ax_r.plot(g['x'], g['rate'], 'o-', color='crimson')
    ax_r.axhline(base_rate, ls='--', color='grey')
    ax_r.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax_r.set_title(f'{col} – fraud rate by quantile bin')

plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(12,5))
sns.countplot(
    data=df,
    x='card_age_months',
    hue='is_fraud',
    # log_scale=True
)

plt.tight_layout()

# Feature Engineering

In [0]:
cat_cols = list(df.select_dtypes(include=["object"]).columns)+ ['day_of_week']
cat_cols.remove('source')

for col in cat_cols:
    mean_order = (
        df.groupby(by=f"{col}")[TARGET]
        .mean()
        .sort_values(ascending=False)
    )

In [0]:
(df.groupby(by=f"merchant_category")[TARGET]
.mean()
.sort_values(ascending=True)).reset_index()[''].rank()

In [0]:
def feature_engineering(fdf: pd.DataFrame) -> pd.DataFrame:
    pass